Import Libraries

In [3]:
import gymnasium as gym
import numpy as np
import random
import ale_py
from gymnasium.envs.registration import make, pprint_registry, register, registry, spec
from IPython.display import clear_output
import time

Initialize Gym Variables

In [4]:
# gym.register_envs(ale_py)
# env_name = "ALE/Pong-v5" # Wont render in ipynb

# env_name = "MountainCar-v0" # Example Discrete
# env_name = "MountainCarContinuous-v0" # Example Continuous


# env_name = "FrozenLake-v1"
try:
    register(
        id="FrozenLakeNoSlip-v1",
        entry_point="gymnasium.envs.toy_text.frozen_lake:FrozenLakeEnv",
        kwargs={"map_name": "4x4", "is_slippery": False},
        max_episode_steps=100,
        reward_threshold=0.78  # optimum = 0.74
    )
except:
    pass
env_name = "FrozenLakeNoSlip-v1"

render_mode_name = "human"
env = gym.make(env_name, render_mode = render_mode_name)


print("Observation Space: ", env.observation_space)
print("Action Space: ", env.action_space, " as type ", type(env.action_space))

Observation Space:  Discrete(16)
Action Space:  Discrete(4)  as type  <class 'gymnasium.spaces.discrete.Discrete'>


Define Agent

In [5]:
class Agent():
    def __init__(self, env):
        self.is_discrete = type(env.action_space) == gym.spaces.discrete.Discrete
        print("Is Discrete? ", self.is_discrete)

        if self.is_discrete:
            self.action_size = env.action_space.n
            print("Action size:", self.action_size)
        else:
            self.action_space_low = env.action_space.low
            self.action_space_high = env.action_space.high
            self.action_shape = env.action_space.shape
            print("Action range:", self.action_space_low, self.action_space_high)
    
    
    def get_action(self, observation):
        if self.is_discrete:
            action = random.choice(range(self.action_size))
        else:
            action = np.random.uniform(self.action_space_low, self.action_space_high, self.action_shape)

        return action


Define QAgent

In [ ]:
class QAgent(Agent):
    def __init__(self, env, learning_rate=0.01, discount_rate = 0.97, epsilon = 1.0):
        super().__init__(env)
        self.observation_size = env.observation_space.n
        print("Observation size:", self.observation_size)

        self.eps = epsilon
        self.learning_rate = learning_rate
        self.discount_rate = discount_rate
        self.build_model()

    def build_model(self):
        self.q_table = 1e-4*np.random.random([self.observation_size, self.action_size])

    def get_action(self, observation):
        q_observation = self.q_table[observation]
        action_greedy = np.argmax(q_observation)
        action_random = super().get_action(observation)
        return action_random if random.random() < self.eps else action_greedy
    
    def train(self, experience):
        observation, action, next_observation, reward, done = experience
        print(experience)

        # Building Q function
        q_next = self.q_table[next_observation]
        q_next = np.zeros([self.action_size]) if done else q_next
        q_target = reward + self.discount_rate * np.max(q_next)

        q_update = q_target - self.q_table[observation,action]
        self.q_table[observation,action] += self.learning_rate * q_update

        if done:
            self.eps = self.eps * 0.99

agent = QAgent(env)


Is Discrete?  True
Action size: 4
Observation size: 16


Training Session

In [7]:

total_reward = 0

for ep in range(100):
    observation, info = env.reset()
    done = False
    while not done:
        action = agent.get_action(observation)
        next_observation, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        agent.train((observation, action, next_observation, reward, done))
        observation = next_observation
        total_reward += reward
        print("s:", observation, "a:", action)
        print("Episode: {}, Total reward: {}, Eps: {}".format(ep, total_reward, agent.eps))

        env.render()
        print(agent.q_table)
    
        time.sleep(0.05)
        clear_output(wait=True)

(0, 2, 1, 0, False)
s: 1 a: 2
Episode: 13, Total reward: 0, Eps: 0.8775210229989678


KeyboardInterrupt: 